In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

2026-09-28 12:21:43,944 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-09-28 12:21:45,397 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-09-28 12:21:48,220 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-09-28 12:21:49,924 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [4]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas7_Areta") \
    .getOrCreate()

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Jumlah Baris Transaksi :", df_transaksi.count())
print("Jumlah Baris Cabang    :", df_cabang.count())
print("Jumlah Baris Menu      :", df_menu.count())

Jumlah Baris Transaksi : 4000
Jumlah Baris Cabang    : 3
Jumlah Baris Menu      : 15


In [6]:
from pyspark.sql.functions import col

df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [9]:
from pyspark.sql.functions import col, row_number, sum as spark_sum
from pyspark.sql.window import Window

ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
    .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+--------------+---------------+---------+
|         nama_cabang|nama_menu| kategori_menu|total_penjualan|peringkat|
+--------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|Makanan Ringan|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|Makanan Ringan|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|      Non-Kopi|        8160000|        2|
+--------------------+---------+--------------+---------------+---------+



In [10]:

df_gabungan.createOrReplaceTempView("transaksi_kopi")

kontribusi_barista = spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM transaksi_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

kontribusi_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [11]:

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

path_output = "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_output)

# Verifikasi dengan membaca kembali
df_verifikasi = spark.read.parquet(path_output)
print("Verifikasi Berhasil! Jumlah baris data yang dibaca ulang:", df_verifikasi.count())
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

2026-09-28 12:30:40,296 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Verifikasi Berhasil! Jumlah baris data yang dibaca ulang: 4000
2026-09-28 12:30:44,638 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 4 items
-rw-r--r--   3 aret supergroup          0 2026-09-28 12:30 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - aret supergroup          0 2026-09-28 12:30 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - aret supergroup          0 2026-09-28 12:30 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - aret supergroup          0 2026-09-28 12:30 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [13]:
from pyspark.sql.functions import avg, count, sum as spark_sum

ringkasan_eksekutif = df_gabungan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi"),
).orderBy(col("total_penjualan").desc())

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+

